# 01 · Collect + download

`collect` expands every playlist / channel / video link into **unique** video IDs (re-running picks up new links and new playlist items).
`download` fetches the best audio stream and standardizes it to **mono 24 kHz MP3** (configurable). Both are resumable.

If YouTube answers *"Sign in to confirm you're not a bot"*, export `cookies.txt` from a logged-in browser and set `download.cookies_file`.

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("collect")

In [ ]:
videos = pd.DataFrame(nb.read_jsonl(lay.videos))
print(len(videos), "videos,", round(videos["duration"].fillna(0).sum() / 3600, 1), "hours listed")
display(videos.groupby("channel")["duration"].agg(["count", lambda s: round(s.sum() / 3600, 1)]).rename(columns={"<lambda_0>": "hours"}).sort_values("hours", ascending=False).head(30))

In [ ]:
nb.run("download")          # add "--limit", "5" to try a few first

In [ ]:
fails = pd.DataFrame(nb.read_jsonl(lay.download_failures))
print(len(fails), "failures"); display(fails.head(20))

In [ ]:
vid = videos["id"].iloc[0]
nb.play(str(lay.raw_audio(vid, cfg.download.format)), f"**{videos['title'].iloc[0]}** (first 30 s shown in the player)")